# YOLOv11 - Uso de Modelos de Deteccao para Identificacao de Barreiras de Acessibilidade em Ambientes Academicos
Notebook responsável por receber um dataset de barreiras arquitetônicas (curb, stairs, handrail, ramp) e utilizar o modelo YOLOv11 para detectar as barreiras em ambientes acadêmicos

In [ ]:
# Instala Ultralytics
%pip install -q ultralytics

# Realiza imports necessários
from google.colab import drive
drive.mount('/content/drive')
import os
import sys
import json
import shutil
import zipfile
import subprocess
from pathlib import Path
import torch
import yaml
from ultralytics import YOLO

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Seleciona o melhor modelo treinado (best.pt), caso já tenha treinado algum modelo
BEST_PT = Path(
    "/content/drive/MyDrive/resultados_yolov11/weights/best.pt"
)

assert BEST_PT.is_file(), f"Arquivo não encontrado: {BEST_PT}"

print("Tamanho:", f"{BEST_PT.stat().st_size / (1024**2):.2f} MB")

try:
    model = YOLO(str(BEST_PT))
    print("\nModelo carregado com sucesso!")
    print("Quantidade de classes:", model.model.nc)
    print("Classes do modelo:", model.names)
    print("Tarefa:", model.task)
    print("GPU disponível:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))

except Exception as e:
    print("\nErro ao carregar o modelo:")
    print(type(e).__name__, str(e))
    raise


Tamanho: 38.63 MB

Modelo carregado com sucesso!
Quantidade de classes: 4
Classes do modelo: {0: 'curb', 1: 'stairs', 2: 'handrail', 3: 'ramp'}
Tarefa: detect
GPU disponível: False


In [ ]:
# Define diretórios
DRIVE = Path("/content/drive/MyDrive")
ZIP_DATASET = DRIVE / "dataset_final.zip"
PASTA_RESULTADOS = DRIVE / "resultados_yolov11"

# Diretório temporário para extração
PASTA_EXTRACAO = Path("/content/dataset_yolov11")

In [ ]:
# Verificar GPU disponível
if torch.cuda.is_available():
    DEVICE = 0
    print("GPU disponível:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    DEVICE = "cpu"
    print("AVISO: nenhuma GPU foi detectada.")
    print("O treinamento será executado em CPU e poderá demorar.")

print("Dispositivo selecionado:", DEVICE)

AVISO: nenhuma GPU foi detectada.
O treinamento será executado em CPU e poderá demorar.
Dispositivo selecionado: cpu


In [ ]:
# Preparar pasta de resultados
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

# Salvar informações da execução
configuracao = {
    "modelo": "YOLOv11m",
    "epocas": 100,
    "imgsz": 640,
    "device": str(DEVICE),
    "dataset": str(ZIP_DATASET),
    "treinamento": "images/train",
    "validacao": "images/val",
    "teste": "images/test",
    "classes": {
        0: "curb",
        1: "stairs",
        2: "handrail",
        3: "ramp"
    }
}

with open(
    PASTA_RESULTADOS / "configuracao_experimento.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(configuracao, f, indent=4, ensure_ascii=False)

In [ ]:
# Verificar e extrair dataset_final.zip
if not ZIP_DATASET.is_file():
    raise FileNotFoundError(
        f"Não encontrei {ZIP_DATASET}. "
        "Confirme se dataset_final.zip está na raiz do Meu Drive."
    )

if not zipfile.is_zipfile(ZIP_DATASET):
    raise ValueError("O arquivo dataset_final.zip não é um ZIP válido.")

# Extrair novamente para garantir que os dados estejam atualizados
if PASTA_EXTRACAO.exists():
    shutil.rmtree(PASTA_EXTRACAO)

PASTA_EXTRACAO.mkdir(parents=True)

print("\nExtraindo dataset_final.zip...")
with zipfile.ZipFile(ZIP_DATASET, "r") as z:
    z.extractall(PASTA_EXTRACAO)

# Localizar a raiz que contém data.yaml e images/ e labels/
RAIZES_POSSIVEIS = [
    PASTA_EXTRACAO,
    *PASTA_EXTRACAO.rglob("*")
]

RAIZ_DATASET = None

for pasta in RAIZES_POSSIVEIS:
    if not pasta.is_dir():
        continue

    if (
        (pasta / "data.yaml").is_file()
        and (pasta / "images" / "train").is_dir()
        and (pasta / "images" / "val").is_dir()
        and (pasta / "images" / "test").is_dir()
        and (pasta / "labels" / "train").is_dir()
        and (pasta / "labels" / "val").is_dir()
        and (pasta / "labels" / "test").is_dir()
    ):
        RAIZ_DATASET = pasta
        break

if RAIZ_DATASET is None:
    raise FileNotFoundError(
        "Estrutura esperada não encontrada dentro do ZIP.\n"
        "Esperado: data.yaml, images/train, images/val, "
        "images/test, labels/train, labels/val e labels/test."
    )

print("Raiz do dataset:", RAIZ_DATASET)


Extraindo dataset_final.zip...
Raiz do dataset: /content/dataset_yolov11/dataset_final


In [ ]:
# Validar data.yaml e suas classes
CAMINHO_YAML = RAIZ_DATASET / "data.yaml"

with open(CAMINHO_YAML, "r", encoding="utf-8") as f:
    dados_yaml = yaml.safe_load(f)

CLASSES_ESPERADAS = {
    0: "curb",
    1: "stairs",
    2: "handrail",
    3: "ramp"
}

nomes_yaml = dados_yaml.get("names", {})

# Aceitar nomes de classes em formato de lista ou dicionário
if isinstance(nomes_yaml, list):
    nomes_normalizados = {
        i: nome for i, nome in enumerate(nomes_yaml)
    }
else:
    nomes_normalizados = {
        int(k): v for k, v in nomes_yaml.items()
    }

if nomes_normalizados != CLASSES_ESPERADAS:
    raise ValueError(
        "As classes do data.yaml não correspondem às esperadas.\n"
        f"Encontradas: {nomes_normalizados}\n"
        f"Esperadas: {CLASSES_ESPERADAS}"
    )

# Garantir que o caminho do YAML corresponda à extração atual
dados_yaml["path"] = str(RAIZ_DATASET)
dados_yaml["train"] = "images/train"
dados_yaml["val"] = "images/val"
dados_yaml["test"] = "images/test"

with open(CAMINHO_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        dados_yaml,
        f,
        sort_keys=False,
        allow_unicode=True
    )

In [ ]:
# Contar imagens de cada divisão
EXTENSOES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def contar_imagens(pasta):
    return sum(
        1 for p in pasta.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSOES
    )

contagens = {}

for split in ["train", "val", "test"]:
    n_imagens = contar_imagens(RAIZ_DATASET / "images" / split)
    n_labels = len(list(
        (RAIZ_DATASET / "labels" / split).glob("*.txt")
    ))

    contagens[split] = {
        "imagens": n_imagens,
        "labels": n_labels
    }

    print(
        f"{split.upper()}: {n_imagens} imagens | "
        f"{n_labels} arquivos de rótulo"
    )

    if n_imagens == 0:
        raise ValueError(f"A divisão {split} não contém imagens.")

    if n_imagens != n_labels:
        raise ValueError(
            f"A divisão {split} tem quantidades diferentes "
            "de imagens e rótulos. Corrija antes de treinar."
        )

# Salvar contagens
with open(
    PASTA_RESULTADOS / "contagens_dataset.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(contagens, f, indent=4, ensure_ascii=False)

TRAIN: 8210 imagens | 8210 arquivos de rótulo
VAL: 2224 imagens | 2224 arquivos de rótulo
TEST: 491 imagens | 491 arquivos de rótulo


In [ ]:
# Execução do treinamento do YOLOv11m no dataset
print('Iniciando treinamento do modelo YOLOv11m...')

# Pesos pré-treinados do YOLO11m
model = YOLO("yolo11m.pt")

resultado_treino = model.train(
    data=str(CAMINHO_YAML),
    epochs=100,
    imgsz=640,
    batch=-1 if DEVICE != "cpu" else 4, # GPU: Ultralytics estima automaticamente; CPU: 4 imagens por lote
    device=DEVICE,
    workers=2,
    patience=20,
    save=True,
    save_period=10,
    plots=True,
    project=str(PASTA_RESULTADOS),
    name="treinamento_yolov11m",
    exist_ok=True,
    verbose=True,
    seed=42
)

In [ ]:
# Execução da validação do YOLOv11m no dataset
print('Iniciando validação do modelo YOLOv11m...')

modelo_validacao = YOLO(str(BEST_PT))

metricas_validacao = modelo_validacao.val(
    data=str(CAMINHO_YAML),
    split="val",
    imgsz=640,
    batch=4,
    device=DEVICE,
    workers=2,
    plots=True,
    project=str(PASTA_RESULTADOS),
    name="validacao_final",
    exist_ok=True,
    verbose=True
)

resumo_validacao = {
    "precisao": float(metricas_validacao.box.mp),
    "recall": float(metricas_validacao.box.mr),
    "mAP50": float(metricas_validacao.box.map50),
    "mAP50_95": float(metricas_validacao.box.map)
}

with open(
    PASTA_RESULTADOS / "metricas_validacao.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(resumo_validacao, f, indent=4, ensure_ascii=False)

print("\nMétricas da validação:")
for nome, valor in resumo_validacao.items():
    print(f"{nome}: {valor}")


Iniciando validação do modelo YOLOv11m...
Ultralytics 8.4.208 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11m summary (fused): 125 layers, 20,033,116 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 793.8±980.1 MB/s, size: 1217.7 KB)
val: Scanning /content/dataset_yolov11/dataset_final/labels/val.cache... 2224 images, 413 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2224/2224 444.2Mit/s 0.0s
val: /content/dataset_yolov11/dataset_final/images/val/handrail_000941.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 3% ──────────── 19/556 5.5s/it 1:13<49:14


KeyboardInterrupt: 

In [ ]:
# Execução do teste do YOLOv11m no dataset

modelo_final = YOLO(str(BEST_PT))

metricas_teste = modelo_final.val(
    data=str(CAMINHO_YAML),
    split="test",
    imgsz=640,
    batch= 4,
    device=DEVICE,
    workers=2,
    plots=True,
    save_json=True,
    project=str(PASTA_RESULTADOS),
    name="teste_mackenzie",
    exist_ok=True
)

resumo_teste = {
    "precisao": float(metricas_teste.box.mp),
    "recall": float(metricas_teste.box.mr),
    "mAP50": float(metricas_teste.box.map50),
    "mAP50_95": float(metricas_teste.box.map)
}

with open(
    PASTA_RESULTADOS / "metricas_teste.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(resumo_teste, f, indent=4, ensure_ascii=False)

print("\nMétricas do teste:")
for nome, valor in resumo_teste.items():
    print(f"{nome}: {valor}")

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 125 layers, 20,033,116 parameters, 0 gradients, 67.8 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 26.7±3.7 MB/s, size: 43.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/dataset_yolov11/dataset_final/dataset_final/dataset_final/labels/test... 491 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 491/491 907.1it/s 0.5s
val: New cache created: /content/dataset_yolov11/dataset_final/dataset_final/dataset_final/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 123/123 4.2it/s 29.2s
                   all        491       1506      0.522      0.375      0.358      0.201
                stairs        364        485      0.706      0.362      0.414      0.253
    